# 1 — M3C 3-φ: Fast SVM no Plano lgγ

> **Objetivo**: introduzir a modulação por vetores espaciais rápida do
> M3C (Gili 2024 Sec 3.2). Mostrar a transformação `abc → lgγ` que
> gera vetores *inteiros*, as 4 candidatas adjacentes (Eqs 26a-d),
> a escolha de triângulo (Eq 28, com a correção de sinal documentada)
> e as razões cíclicas (Eqs 29-30).

**Referências da tese**
* Sec 3.2 — Fast SVM no plano lgγ
* Eq 25 — matriz de transformação (gera vetores inteiros)
* Eqs 26a-d — 4 vetores adjacentes (`V_ul`, `V_lu`, `V_ll`, `V_uu`)
* Eq 28 — escolha entre triângulos (typo corrigido no código)
* Eqs 29-30 — razões cíclicas


In [ ]:
import sys, os
from pathlib import Path
_HERE = Path.cwd() / "projects" / "inverters" / "m3c_3phase"
if str(_HERE) not in sys.path:
    sys.path.insert(0, str(_HERE))

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 100

In [ ]:
from m3c_3phase_model import (
    M3cParams, LG_TRANSFORM_MATRIX, abc_to_lg, lg_to_abc,
    fast_svm_4_vectors, fast_svm_pick_triangle, fast_svm_duty_cycles,
    make_fast_svm_fn,
)

params = M3cParams()
print(f"M3cParams (Tab. 15 defaults):")
print(f"  N_SM           = {params.n_sm_per_module}")
print(f"  V_cap nominal  = {params.v_cap_nominal} V")
print(f"  Níveis L-L     = {params.n_levels_LL}")
print(f"  f_in / f_out   = {params.f_in} / {params.f_out} Hz")


## 1.1 — Matriz de transformação (Eq 25)

A transformação `lgγ` é uma matriz **inteira**:
$$
\begin{bmatrix} V_l \\ V_g \\ V_\gamma \end{bmatrix} =
\begin{bmatrix} 1 & -1 & 0 \\ 0 & 1 & -1 \\ 1 & 1 & 1 \end{bmatrix}
\begin{bmatrix} V_a \\ V_b \\ V_c \end{bmatrix}
$$

Propriedade central: **entrada inteira → saída inteira**. Sem
trigonometria. Implementável em FPGA com somadores apenas.


In [ ]:
print("LG_TRANSFORM_MATRIX:")
print(LG_TRANSFORM_MATRIX)

# Demo: vetor abc inteiro → lgγ inteiro.
abc = np.array([2, -1, 1])
lgg = abc_to_lg(abc)
print(f"\nabc = {abc.tolist()} → lgγ = {lgg.tolist()}")
print(f"Inverso: lgγ → abc: {lg_to_abc(lgg).tolist()}")


## 1.2 — Os 4 vetores adjacentes (Eqs 26a-d)

Dado um ponto de referência `(V_ref_l, V_ref_g)` no plano lgγ, a
Fast SVM identifica os 4 vértices inteiros do quadrilátero que o
contém:

* `V_ul = (⌈l⌉, ⌊g⌋)` — upper-l, lower-g
* `V_lu = (⌊l⌋, ⌈g⌉)` — lower-l, upper-g
* `V_ll = (⌊l⌋, ⌊g⌋)` — lower-l, lower-g (triângulo S)
* `V_uu = (⌈l⌉, ⌈g⌉)` — upper-l, upper-g (triângulo N)


In [ ]:
# Visualização: percorrer o quadrilátero ao redor de (1.7, 0.4).
v_ref_l, v_ref_g = 1.7, 0.4
V_ul, V_lu, V_ll, V_uu = fast_svm_4_vectors(v_ref_l, v_ref_g)

print(f"Referência: ({v_ref_l}, {v_ref_g})")
print(f"V_ul = {V_ul}, V_lu = {V_lu}")
print(f"V_ll = {V_ll}, V_uu = {V_uu}")

picked = fast_svm_pick_triangle(v_ref_l, v_ref_g)
print(f"\nTriângulo escolhido: {picked} (S = V_ll, N = V_uu)")

# Plot.
fig, ax = plt.subplots(figsize=(6,6))
for v, label in [(V_ul, "V_ul"), (V_lu, "V_lu"), (V_ll, "V_ll"), (V_uu, "V_uu")]:
    ax.plot(*v, "o", markersize=12)
    ax.annotate(label, v, textcoords="offset points", xytext=(8, 8))
ax.plot(v_ref_l, v_ref_g, "r*", markersize=20, label="V_ref")
# Sketch the chosen triangle.
if picked == "ll":
    tri = [V_ul, V_lu, V_ll, V_ul]
else:
    tri = [V_ul, V_lu, V_uu, V_ul]
tri_arr = np.array(tri)
ax.plot(tri_arr[:,0], tri_arr[:,1], "g--", alpha=0.6, label=f"Triângulo {picked}")
ax.set_xlabel("l")
ax.set_ylabel("g")
ax.set_title("4 vetores adjacentes na malha lgγ")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()


## 1.3 — Razões cíclicas — Eqs 29-30

As razões cíclicas (`δ_ul`, `δ_lu`, `δ_third`) são *bilineares* nas
coordenadas (l, g). Cada razão deve estar em `[0, 1]` e a soma `≤ 1`.

Aqui mostramos a varredura completa de razões cíclicas para
`(V_ref_l, V_ref_g) ∈ [-N, +N]²`:


In [ ]:
# Heatmap das razões cíclicas no plano lgγ.
N = params.n_sm_per_module
l_grid = np.linspace(-N, N, 51)
g_grid = np.linspace(-N, N, 51)
delta_sum = np.zeros((51, 51))
for i, l in enumerate(l_grid):
    for j, g in enumerate(g_grid):
        d_ul, d_lu, d_third, _label = fast_svm_duty_cycles(l, g)
        delta_sum[j, i] = d_ul + d_lu + d_third

fig, ax = plt.subplots(figsize=(8,7))
im = ax.imshow(delta_sum, origin="lower", extent=(-N, N, -N, N),
                cmap="viridis", vmin=0, vmax=1.05, aspect="equal")
plt.colorbar(im, ax=ax, label="δ_ul + δ_lu + δ_third")
ax.set_xlabel("V_ref_l")
ax.set_ylabel("V_ref_g")
ax.set_title("Soma das razões cíclicas — deve ser ≤ 1 em todo o plano")
plt.tight_layout()
plt.show()

print(f"\nDuty-sum stats: min={delta_sum.min():.4f}, max={delta_sum.max():.4f}")
print(f"(esperado: min=0 nos vértices, max=1 nas arestas)")


## 1.4 — Resumo

* A transformação lgγ gera vetores **inteiros**, sem trigonometria.
* Para qualquer ponto de referência, há 4 vetores inteiros
  adjacentes (`V_ul`, `V_lu`, `V_ll`, `V_uu`).
* A escolha do triângulo (S ou N) é geométrica (Eq 28 corrigida).
* As razões cíclicas são bilineares e satisfazem `Σδ ≤ 1`.

Próximo notebook: cálculo das tensões dos 9 módulos a partir das
referências SVM e seleção de configuração via função custo.
